# HW04-ICA :: Part A (PyTorch CNN on MNIST)

COSC 6373

Adam Nelson-Archer, 2140122


In [ ]:
from __future__ import annotations

import os
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms


SEED = 6373
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Reproducibility
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Device:", device)
print("CWD:", os.getcwd())

c:\Program Files\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.2.2+cpu
CUDA available: False
Device: cpu
CWD: c:\Users\Adam\Desktop\projects_and_code\School\COSC6373\HW4\Part_A


In [2]:
# Paths (relative to this notebook folder)
NOTEBOOK_DIR = Path.cwd()
DATA_DIR = (NOTEBOOK_DIR / "data").resolve()

print("DATA_DIR:", DATA_DIR)

DATA_DIR: C:\Users\Adam\Desktop\projects_and_code\School\COSC6373\HW4\Part_A\data


In [3]:
# MNIST data loaders
# Standard MNIST normalization values
transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,)),
    ]
)

train_ds = datasets.MNIST(root=DATA_DIR, train=True, download=True, transform=transform)
test_ds = datasets.MNIST(root=DATA_DIR, train=False, download=True, transform=transform)

BATCH_SIZE = 64
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)

print("Train size:", len(train_ds))
print("Test size:", len(test_ds))
x0, y0 = train_ds[0]
print("One sample tensor shape:", tuple(x0.shape), "label:", y0)

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 9912422/9912422 [00:00<00:00, 17816125.83it/s]


Extracting C:\Users\Adam\Desktop\projects_and_code\School\COSC6373\HW4\Part_A\data\MNIST\raw\train-images-idx3-ubyte.gz to C:\Users\Adam\Desktop\projects_and_code\School\COSC6373\HW4\Part_A\data\MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 28881/28881 [00:00<00:00, 804412.63it/s]


Extracting C:\Users\Adam\Desktop\projects_and_code\School\COSC6373\HW4\Part_A\data\MNIST\raw\train-labels-idx1-ubyte.gz to C:\Users\Adam\Desktop\projects_and_code\School\COSC6373\HW4\Part_A\data\MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 1648877/1648877 [00:00<00:00, 6528336.69it/s]


Extracting C:\Users\Adam\Desktop\projects_and_code\School\COSC6373\HW4\Part_A\data\MNIST\raw\t10k-images-idx3-ubyte.gz to C:\Users\Adam\Desktop\projects_and_code\School\COSC6373\HW4\Part_A\data\MNIST\raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 4542/4542 [00:00<?, ?it/s]

Extracting C:\Users\Adam\Desktop\projects_and_code\School\COSC6373\HW4\Part_A\data\MNIST\raw\t10k-labels-idx1-ubyte.gz to C:\Users\Adam\Desktop\projects_and_code\School\COSC6373\HW4\Part_A\data\MNIST\raw



Train size: 60000
Test size: 10000
One sample tensor shape: (1, 28, 28) label: 5


In [ ]:
# Quick check of data plumbing
images, labels = next(iter(train_loader))
print("Batch images:", tuple(images.shape))
print("Batch labels:", tuple(labels.shape))
print("First 10 labels:", labels[:10].tolist())

Batch images: (64, 1, 28, 28)
Batch labels: (64,)
First 10 labels: [1, 1, 1, 8, 9, 8, 4, 4, 3, 3]


In [5]:
class BaseMNISTCNN(nn.Module):
    """Base CNN for MNIST.

    Required pipeline (HW spec):
    Convolutional Layer -> MaxPooling -> Flatten -> Dropout -> Fully Connected layer
    """

    def __init__(self, num_classes: int = 10):
        super().__init__()

        # 1x28x28 -> 16x24x24 (kernel=5, no padding)
        self.conv = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=5)

        # 16x24x24 -> 16x12x12
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Flatten: 16 * 12 * 12 = 2304
        self.dropout = nn.Dropout(p=0.5)
        self.fc = nn.Linear(16 * 12 * 12, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.conv(x)
        x = F.relu(x)
        x = self.pool(x)
        x = torch.flatten(x, start_dim=1)
        x = self.dropout(x)
        x = self.fc(x)
        return x


model = BaseMNISTCNN().to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print("Trainable parameters:", f"{n_params:,}")

BaseMNISTCNN(
  (conv): Conv2d(1, 16, kernel_size=(5, 5), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=2304, out_features=10, bias=True)
)
Trainable parameters: 23,466


In [6]:
# check: run one forward pass
model.eval()
images, labels = next(iter(train_loader))
logits = model(images.to(device))
print("Logits shape:", tuple(logits.shape))  # (batch_size, 10)
print("Example predicted class (first item):", int(logits.argmax(dim=1)[0].cpu()))

Logits shape: (64, 10)
Example predicted class (first item): 5


## Model Summary

Base CNN model:
- Convolutional Layer: `Conv2d(1 → 16, kernel_size=5)` + ReLU
- MaxPooling: `MaxPool2d(2×2)`
- Flatten: `16×12×12 = 2304`
- Dropout: `p=0.5`
- Fully Connected: `Linear(2304 → 10)` (logits)


## Acknowledgment

I used a coding assistant (ChatGPT, GPT‑5.2) to help scaffold and organize this notebook.
